# Exercise 08 : A Deep Agent

- `create_deep_agent` includes files, execute and task ; planning with `write_todos` is opt-in
- A backend routes manuals on disk ; a skill is a folder with `SKILL.md`
- A dict subagent receives the task description, not the parent messages

You will build a small Aurelia Deep Agent with planning, a manuals backend, one
skill you write, and one parts subagent. Then you will read the todo list, the
files it wrote, and what the subagent saw. All data is made up.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [ ]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

## Step 1 : Tools and the manuals backend

In [ ]:
import json
import sqlite3
import threading
import time
from pathlib import Path

from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import AIMessage, HumanMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, FilesystemBackend, CompositeBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import InMemorySaver


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1800, max_retries=0)
LIMIT = {"recursion_limit": 36}
DB = Path(data("capstone/service.db")).resolve()
MANUALS = Path(data_folder("capstone/manuals")).resolve()
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}


def ask(text):
    return {"messages": [HumanMessage(text)]}


def answer(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        print("metrics :", json.dumps(row))
        measured.last = row


async def measured_async(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return await operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        print("metrics :", json.dumps(row))
        measured.last = row


def show(result):
    print("tools :", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer :", answer(result))


@tool
def part_stock(part: str) -> str:
    """Look up stock, depot and name for one Aurelia part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


disk = FilesystemBackend(root_dir=MANUALS, virtual_mode=True)
backend = CompositeBackend(default=StateBackend(), routes={"/manuals/": disk})
sample = disk.read("/ct-400-error-codes.md")
sample_text = sample.file_data["content"] if sample.file_data else str(sample.error)
print("manual files :", len(PAGES))
print("sample :", sample_text[:160].replace("\n", " "))
print("disk paths :", [e["path"] for e in disk.ls("/").entries][:5], "...")
print("routed paths :", [e["path"] for e in backend.ls("/manuals/").entries][:5], "...")
print("backend ready")

In [ ]:
print("check :", "PASS" if len(PAGES) >= 20 else "FAIL", "manuals under /manuals/")
assert len(PAGES) >= 20

## Step 2 : Write the skill

Complete the `SKILL.md` body. The front matter needs `name` and `description`.
The body must tell the agent to read under `/manuals/`, state the meaning and
first action, and keep the briefing to three bullets.

In [ ]:
SKILL_MD = """---
name: ____
description: ____
---

____
"""
print(SKILL_MD)

<details><summary>Hint</summary>

The slide *Skills* and module 08 demo06b · skills show the front matter and the instructions below it.
The check cell under this step gives the name, and Step 4 uses the same name as the folder.
The description says when to use the skill. The body states the requirements of the step.

</details>

In [ ]:
assert "name: error-briefing" in SKILL_MD
assert "/manuals/" in SKILL_MD
assert "three" in SKILL_MD.lower()
print("check : PASS skill text")

## Step 3 : The parts subagent

Complete the dict for `parts`. `parts_llm` records the human messages that
model receives, which shows what the subagent saw.

In [ ]:
seen_by_parts = []


class Capture(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        for batch in messages:
            for message in batch:
                if isinstance(message, HumanMessage):
                    seen_by_parts.append(str(message.content)[:300])


parts_llm = make_llm(callbacks=[counter, Capture()], max_tokens=800, max_retries=0)
parts = {
    "name": ____,
    "description": ____,
    "system_prompt": ____,
    "tools": ____,
    "model": parts_llm,
}
print(parts["name"], parts["description"])

<details><summary>Hint</summary>

The slide *Dict subagents* and module 08 demo07 · subagents show the keys of this dict. The check cell under this step gives the name and the tool list. Write a one-line description and a short system prompt.

</details>

In [ ]:
assert parts["name"] == "parts" and parts["tools"] == [part_stock]
print("check : PASS parts subagent")

## Step 4 : Assemble and run

Give `create_deep_agent` the backend, planning middleware, the skill path and
the parts subagent. The task stays small : one todo, one briefing, one delegated
stock lookup and one file write.

In [ ]:
agent = create_deep_agent(
    model=llm,
    backend=____,
    middleware=____,
    skills=____,
    subagents=____,
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You support Aurelia field engineers. Do each step once and stop. "
        "Read the pinned error-briefing skill. Call task with subagent_type parts for "
        "CT-FN-02 stock only. Write one todo and mark it completed. Save /briefing.md. "
        "Do not use the shell. Do not call the general-purpose subagent."
    ),
)
files = {"/skills/error-briefing/SKILL.md": create_file_data(SKILL_MD)}
seen_by_parts.clear()
result = measured("deep-aurelia", lambda: agent.invoke({
    **ask(
        "E-CT-110 briefing and CT-FN-02 stock. One todo, skill, parts task, /briefing.md, "
        "then two short sentences."
    ),
    "files": files,
    "pinned_skills": ["error-briefing"],
}, {**LIMIT, "configurable": {"thread_id": "ex08"}}))
show(result)
print("todos :", result.get("todos"))
print("files :", sorted(result.get("files", {})))
print("seen_by_parts :", seen_by_parts)

<details><summary>Hint</summary>

The slides *Planning with write_todos*, *Skills* and *Dict subagents* with module 08 demo02b · write-todos, module 08 demo06b · skills and module 08 demo07 · subagents show these arguments. `backend` takes the object from Step 1, and the other three take lists. The `skills` path is the parent folder of `error-briefing` in the `files` dict below.

</details>

In [ ]:
import re

text = answer(result).lower()
files_ok = any(path.endswith("briefing.md") or path == "/briefing.md"
               for path in result.get("files", {}))
todos_ok = bool(result.get("todos"))
stock_ok = bool(re.search(r"\b6\b|\bsix\b", text)) and "houten" in text
tools_used = [m.name for m in result["messages"] if m.type == "tool"]
task_ok = "task" in tools_used
seen_ok = bool(seen_by_parts) and not any("error-briefing" in s.lower() and "skill.md" in s.lower()
                                          for s in seen_by_parts)
checks = {"files_ok": files_ok, "todos_ok": todos_ok, "stock_ok": stock_ok,
          "task_ok": task_ok, "seen_ok": seen_ok}
print("check :", "PASS" if all(checks.values()) else "FAIL", checks)
assert all(checks.values()), checks

## If time permits

- Change the skill so the briefing must name the depot as well, and run again
- Add a second dict subagent for manuals and drop the `/manuals/` reads from the parent
- Compare tokens with and without `TodoListMiddleware` on the same task